# relay_grid

> Phase 23 (`specs/multiagent/phase23/`, experiment C1b): the grid. `ARMS23` (C1's `squads`, `coupled` and `bridge1`, and the four new arms `relayEvid`, `bridgeRole`, `bridgeFix` and `mirrorWindow`), `C1B_CELLS` (23 new boards with independent liars) and `C1B_CLEAN` (10 clean), `PILOT23` (seed 1890), `REUSED23` (36 of C1's boards, replayed from the cache at $0); `c1bCell`, `c1bGrid`, `c1bPilotGrid`, `reusedCells23`, and the cost before the pilot, `c1bRange`, from C1's measured cost per board.

In [ ]:
#| default_exp relay_grid

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from explRL.bridge_grid import ARMS20, N20, SEEDS5, SEEDS8, c1Grid, reusedCells20
from explRL.clue_colony import checkOptions
from explRL.colony_grid import PILOT_SEED, colonyCell
from explRL.fix_grid import cellCalls

_C = ARMS20['coupled']
ARMS23 = dict(squads=ARMS20['squads'], coupled=_C, bridge1=ARMS20['bridge1'],
              relayEvid=dict(_C, bridges=1, evidence=True), bridgeRole=dict(_C, bridges=1, bridgeRole=True),
              bridgeFix=dict(_C, bridges=1, evidence=True, bridgeRole=True), mirrorWindow=dict(_C, bridges=0, mirrorWindow=True))
NEW23 = ('relayEvid', 'bridgeRole', 'bridgeFix', 'mirrorWindow')
C1B_CELLS = dict(relayEvid=SEEDS5, bridgeRole=SEEDS5, bridgeFix=SEEDS8, mirrorWindow=SEEDS5)   # 23 boards at N 64, independent liars
C1B_CLEAN = dict(bridgeFix=SEEDS5, mirrorWindow=SEEDS5)                                      # 10 clean boards
PILOT23 = NEW23                                                                               # seed 1890, with liars
REUSED23 = dict(liars=dict(squads=SEEDS8, coupled=SEEDS5, bridge1=SEEDS8), clean=dict(squads=SEEDS5, coupled=SEEDS5, bridge1=SEEDS5))
PER_BOARD23, ALLOW23 = dict(independent=0.65, clean=0.61), 1.15    # C1's measured cost per board (bridge arms; clean); the allowance
arm23 = lambda c: c['level'].split('|')[0]


def c1bCell(study, arm, seed, fault='independent', runner='llm'):
    "One C1b cell at N 64, level '<arm>|<fault>' (as C1's c1Cell, on ARMS23), checked."
    c = colonyCell(study, arm, N20, seed, fault, runner, arms=ARMS23)
    checkOptions(c['cfg']['world'], N20, c['cfg'].get('nLiar', 0))
    return c


def c1bGrid():
    "Study colony23: C1B_CELLS with independent liars (23) and C1B_CLEAN (10): the 33 new boards."
    return [c1bCell('colony23', a, s, f) for cs, f in ((C1B_CELLS, 'independent'), (C1B_CLEAN, 'clean'))
            for a, ss in cs.items() for s in ss]


def c1bPilotGrid():
    "Study colony23Pilot: PILOT23 at N 64 with independent liars, seed 1890 (not analysed)."
    return [c1bCell('colony23Pilot', a, PILOT_SEED) for a in PILOT23]


def reusedCells23():
    """C1's 36 boards that C1b reuses (FR-C2): coupled and bridge1 with liars and clean, and squads clean, from c1Grid; squads
    with liars from Stage 4 (reusedCells20). Their saved rows are C1's runs.csv and reused.csv."""
    want = {(f'{a}|{f}', s) for f, cs in (('independent', REUSED23['liars']), ('clean', REUSED23['clean'])) for a, ss in cs.items() for s in ss}
    out = [c for c in c1Grid() + reusedCells20() if (c['level'], c['seed']) in want]
    assert len(out) == len(want), (len(out), len(want))
    return out


def c1bRange(per=PER_BOARD23, allow=ALLOW23):
    """README's cost range before the pilot: (low, high) for the main run, the pilot and both, from C1's cost per board spread
    over cellCalls (1,920 calls per board with liars, 2,560 clean); high: x allow (the role and evidence text)."""
    calls = dict(independent=1920, clean=2560)
    rate = lambda c: per[c['level'].split('|')[1]] / calls[c['level'].split('|')[1]]
    cost = lambda g, x: round(sum(cellCalls(c) * rate(c) * x for c in g()), 2)
    r = {k: (cost(g, 1.0), cost(g, allow)) for k, g in (('main', c1bGrid), ('pilot', c1bPilotGrid))}
    r['total'] = tuple(round(m + p, 2) for m, p in zip(r['main'], r['pilot']))
    return dict(r, boards=len(c1bGrid()) + len(c1bPilotGrid()), calls=sum(cellCalls(c) for c in c1bGrid()))

In [ ]:
# test_c1b_grid (FR-C1, FR-C2): 23 + 10 new boards on README's seeds, 4 pilot boards, 36 reused; the arms' worlds pass
# checkOptions and carry exactly C1's world plus their keys; the cost range is README's ($23.7-27.3)
from collections import Counter
from explRL.relay_board import RELAY_KEYS
g, p, r = c1bGrid(), c1bPilotGrid(), reusedCells23()
assert len(g) == 33 and len(p) == 4 and len(r) == 36
assert Counter(c['level'] for c in g) == {'relayEvid|independent': 5, 'bridgeRole|independent': 5, 'bridgeFix|independent': 8,
                                         'mirrorWindow|independent': 5, 'bridgeFix|clean': 5, 'mirrorWindow|clean': 5}
assert {c['seed'] for c in g if c['level'] == 'bridgeFix|independent'} == set(range(1800, 1808))
assert all(c['seed'] in range(1800, 1805) for c in g if c['level'] != 'bridgeFix|independent')
assert {c['seed'] for c in p} == {1890} and {arm23(c) for c in p} == set(NEW23) and all(c['study'] == 'colony23Pilot' for c in p)
assert Counter(c['level'] for c in r) == {'squads|independent': 8, 'coupled|independent': 5, 'bridge1|independent': 8,
                                         'squads|clean': 5, 'coupled|clean': 5, 'bridge1|clean': 5}
assert {c['study'] for c in r} == {'colony20', 'colony4'}
c1 = {(c['level'], c['seed']): c for c in c1Grid()}
for c in g:
    w, base = c['cfg']['world'], c1[('bridge1|' + c['level'].split('|')[1], c['seed'])]['cfg']['world']
    extra = {k: v for k, v in w.items() if k not in base or base[k] != v}
    assert extra == {k: True for k in RELAY_KEYS if ARMS23[arm23(c)].get(k)} | ({'bridges': 0} if arm23(c) == 'mirrorWindow' else {}), (c['level'], extra)
    assert c['cfg']['nLiar'] == (16 if c['level'].endswith('independent') else 0) and c['N'] == 64
cr = c1bRange()
assert cr['boards'] == 37 and abs(cr['total'][0] - 23.65) < 0.01 and abs(cr['total'][1] - 27.2) < 0.01, cr
assert abs(cr['pilot'][0] - 2.6) < 0.01 and cr['calls'] == 23 * 1920 + 10 * 2560
print(cr)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()